# 08. 메타분석 1 — 키니스(keyness): 언어 집합과 2026년에 유의하게 더 쓰이는 말 (로그 오즈비, Dirichlet 사전분포)

- **목적**: 어휘 전체를 대상으로 "이 집합에서 더 쓰이는 말"을 순위로 낸다. Monroe·Colaresi·Quinn(2008)의 정보적 Dirichlet 사전분포 로그 오즈비(log-odds ratio, informative Dirichlet prior)와 z 점수. 연어 분석(07)이 'agent' 주변만 본다면 키니스는 어휘 전체를 본다.
- **비교**: (가) 한국어·중국어·일본어 집합(사람 라벨, D36) 대 나머지 세계, (나) 2026년(1~9월) 대 2023~2025년 세계 전체.
- **입력**: spaCy 표제어 캐시 `analysis/output/cache/lemmas_2023_2026q3.jsonl`(제목+초록, 02와 같은 전처리), `allc`
- **계산**: 집합 i와 j의 단어 빈도 $y_{iw}, y_{jw}$, 사전분포 $\alpha_w = \alpha_0 \cdot n_w / N$(세계 전체 빈도 비례, $\alpha_0 = 1{,}000$). $\delta_w = \log\frac{y_{iw}+\alpha_w}{n_i+\alpha_0-y_{iw}-\alpha_w} - \log\frac{y_{jw}+\alpha_w}{n_j+\alpha_0-y_{jw}-\alpha_w}$, $z_w = \delta_w / \sqrt{1/(y_{iw}+\alpha_w) + 1/(y_{jw}+\alpha_w)}$. 전체 빈도 20 미만인 단어는 뺀다.
- **주의**: 토큰 빈도 기준이라 한 논문에서 반복된 말이 크게 잡힌다(문서 빈도도 함께 적는다). 언어 이름(korean, chinese, japanese)은 집합 정의에 쓰였으므로 결과에서 뺀다. 작은 집합(한국어 25, 일본어 18)은 z가 작아도 순위는 읽을 수 있다.


In [ ]:
import glob, json, re, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "verification").exists())   # 노트북은 analysis/notebooks/ 에 있다
DATA = ROOT / "preprocessing/output"                                   # 입력: 전처리 산출(정제 CSV)
VER = ROOT / "verification"                                            # 수치의 정본 JSON
FIG = ROOT / "analysis/output/figures"; FIG.mkdir(parents=True, exist_ok=True)   # 출력: 노트북 그림. 원고 그림은 paper/figures/
TAB = ROOT / "analysis/output/tables"; TAB.mkdir(parents=True, exist_ok=True)    # 출력: 노트북 표
CACHE = ROOT / "analysis/output/cache"; CACHE.mkdir(parents=True, exist_ok=True) # 출력: 재실행용 캐시(git 제외)
TODAY = pd.Timestamp.today().strftime("%Y%m%d")
SUFFIX = "2023_2026q3"   # D30: 분석 기간 2023-01 ~ 2026-09 말뭉치

try:
    sys.path.insert(0, str(ROOT / "analysis/scripts"))
    import figure_style; figure_style.apply_style(); C = figure_style.COLORS; CYC = figure_style.COLOR_CYCLE
except Exception:
    C = {"green": "#46B07B", "orange": "#FF9E02", "teal": "#3A9E8F", "coral": "#E87B4A", "green_dark": "#2D7A53", "orange_dark": "#C47800", "green_light": "#8DD4AA", "orange_light": "#FFD080", "green_pale": "#C8EBDA", "orange_pale": "#FFECC0", "gray": "#7F8C8D", "gray_light": "#BDC3C7", "gray_dark": "#4A4A4A", "text": "#333333"}; CYC = [C["green"], C["orange"], C["teal"], C["coral"]]
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

LANGS = ["ko", "zh", "ja"]
LANG_NAME = {"ko": "Korean", "zh": "Chinese", "ja": "Japanese"}
YEARS = [2023, 2024, 2025, 2026]          # 2026은 1~9월(D30)
MIN_N = 10   # 투영 집합의 토픽별 건수가 이보다 작으면 lift를 보고하지 않는다

allc = pd.read_csv(DATA / f"corpus_llm_agent_all_{SUFFIX}.csv", dtype={"id": str})
cjk_signal = pd.read_csv(DATA / f"corpus_llm_agent_cjk_lang_{SUFFIX}.csv", dtype={"id": str})   # 키워드 신호 257편 + 사람 라벨(D36)
counts = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
YEAR_BASIS = counts["definitions"]["year_basis"]   # D31: "id" = arXiv id 접두(최초 제출 월)
LABEL_TARGET = "그 언어의 텍스트를 실제 연구 대상으로 다룸"; LABEL_MULTI = "다국어 실험의 한 언어로 포함"
cjk = cjk_signal[cjk_signal.cjk_lang].copy()       # 최종 대상 언어 집합 = 신호 ∧ 라벨 '실제 연구 대상'(D36)

# 정본 대조: 노트북이 읽은 자료가 정본 JSON과 같은지 확인한다
assert YEAR_BASIS == "id", "연도 기준이 D31(id 접두)이 아니다"
assert len(allc) == counts["pipeline"]["llm_agent_all"], "전체 건수가 정본과 다르다"
assert len(cjk_signal) == counts["pipeline"]["llm_agent_cjk_signal"], "대상 언어 신호 건수가 정본과 다르다"
assert len(cjk) == counts["pipeline"]["llm_agent_cjk_lang"], "대상 언어 집합 건수가 정본과 다르다"
assert (cjk_signal.lang_label != "").all(), "라벨 없는 신호 논문이 있다"
assert allc.groupby("year").size().to_dict() == {int(k): v for k, v in counts["llm_agent_by_year"].items()}
assert set(cjk.id) <= set(allc.id) and allc.id.is_unique
for l in LANGS:
    allc[f"lang_{l}"] = allc[f"lang_{l}"].astype(bool); allc[f"lang_{l}_signal"] = allc[f"lang_{l}_signal"].astype(bool)
assert {l: int(allc[f"lang_{l}"].sum()) for l in LANGS} == counts["cjk_lang_by_lang_total"]
N_ALL_BY_YEAR = {int(k): v for k, v in counts["pipeline"]["unique_by_year"].items()}   # 분모: 다섯 cs 분류의 고유 논문 수
print("정본 대조 통과: LLM 에이전트", len(allc), "| 한중일어 신호", len(cjk_signal), "| 대상 언어 집합(사람 라벨)", len(cjk), "| 연도 기준", YEAR_BASIS, "| 기간", counts["definitions"]["period"][:24])

In [ ]:
# 메타분석 공통: spaCy 표제어 캐시(02에서 만든 것)와 LDA 문서 배정을 읽는다
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
DOMAIN_STOP = {"paper", "propose", "proposed", "method", "methods", "result", "results", "show", "shows", "model", "models", "based", "approach", "approaches",
               "task", "tasks", "performance", "llm", "llms", "agent", "agents", "large", "language", "framework", "novel", "existing", "demonstrate",
               "experiments", "experimental", "study", "work", "using", "use", "data", "code", "available", "https", "github", "com", "et", "al"}
STOP = set(ENGLISH_STOP_WORDS) | DOMAIN_STOP
lem = {}
for line in open(CACHE / f"lemmas_{SUFFIX}.jsonl"):
    r = json.loads(line); lem[r["id"]] = [t for t in r["t"] if t not in STOP and len(t) > 2 and not t.isdigit()]
assert set(lem) == set(allc.id), "표제어 캐시와 말뭉치의 id가 다르다"
dt = pd.read_csv(TAB / "lda_doc_topics.csv", dtype={"id": str}); assert len(dt) == len(allc)
allc = allc.merge(dt[["id", "lda_topic", "lda_prob"]], on="id", how="left")
ld = json.loads(Path(sorted(glob.glob(str(VER / "topics_lda_*.json")))[-1]).read_text()); K = ld["K"]
label = {int(t): v["label"] for t, v in ld["topics"].items()}
allc["ym"] = allc.year * 100 + allc.month
MONTHS = sorted(allc.ym.unique()); assert len(MONTHS) == 45 and MONTHS[0] == 202301 and MONTHS[-1] == 202609
print("문서", len(allc), "| 표제어 토큰", sum(len(v) for v in lem.values()), "| 토픽", K, "| 월", len(MONTHS))

## 1. 로그 오즈비 계산 함수와 집합별 빈도

In [ ]:
LANG_WORDS = {"korean", "korea", "chinese", "china", "mandarin", "japanese", "japan", "hangul", "cjk", "english"}
def counts_of(ids):
    c = Counter(); d = Counter()
    for i in ids:
        toks = lem[i]; c.update(toks); d.update(set(toks))
    return c, d
world_c, world_d = counts_of(allc.id); N = sum(world_c.values())
vocab = [w for w, n in world_c.items() if n >= 20 and w not in LANG_WORDS]; print("어휘(빈도 20 이상)", len(vocab))
def log_odds(ci, ni, cj, nj, alpha0=1000.0):
    rows = []
    for w in vocab:
        aw = alpha0 * world_c[w] / N; yi, yj = ci.get(w, 0), cj.get(w, 0)
        d = np.log((yi + aw) / (ni + alpha0 - yi - aw)) - np.log((yj + aw) / (nj + alpha0 - yj - aw))
        z = d / np.sqrt(1 / (yi + aw) + 1 / (yj + aw)); rows.append((w, yi, yj, round(float(d), 3), round(float(z), 2)))
    return pd.DataFrame(rows, columns=["word", "n_in", "n_rest", "log_odds", "z"]).set_index("word")

## 2. 언어 집합 대 나머지 세계 — 상위 20(많이 쓰는 말)과 하위 10(적게 쓰는 말)

- 출력: `analysis/output/tables/keyness_lang_{ko,zh,ja}.csv`, `analysis/output/figures/fig9a_keyness_lang.png`
- 읽는 법: z가 클수록 그 집합에서 상대적으로 많이 쓰인 말. 문서 빈도(df_in)가 2~3편이면 특정 논문의 반복일 수 있다.


In [ ]:
KL = {}
for l in LANGS:
    ids_in = allc[allc[f"lang_{l}"]].id; ids_out = allc[~allc[f"lang_{l}"]].id
    ci, di = counts_of(ids_in); cj, dj = counts_of(ids_out)
    t = log_odds(ci, sum(ci.values()), cj, sum(cj.values())); t["df_in"] = [di.get(w, 0) for w in t.index]; t["df_rest"] = [dj.get(w, 0) for w in t.index]
    t["pct_docs_in"] = (100 * t.df_in / len(ids_in)).round(1); t["pct_docs_rest"] = (100 * t.df_rest / len(ids_out)).round(2)
    t = t.sort_values("z", ascending=False); t.to_csv(TAB / f"keyness_lang_{l}.csv"); KL[l] = t
    print(f"\n{LANG_NAME[l]} (n={len(ids_in)}) 많이 쓰는 말:", ", ".join(f"{w}(z={r.z:.1f}, {int(r.df_in)}편)" for w, r in t.head(20).iterrows()))
    print(f"{LANG_NAME[l]} 적게 쓰는 말:", ", ".join(f"{w}(z={r.z:.1f})" for w, r in t.tail(10).iloc[::-1].iterrows()))
fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))
for ax, l in zip(axes, LANGS):
    t = KL[l].head(20).iloc[::-1]; ax.barh(t.index, t.z, color="#2E86C1"); ax.set_title(f"{LANG_NAME[l]} vs rest (n={int(allc[f'lang_{l}'].sum())})", fontsize=10); ax.set_xlabel("z (log-odds, Dirichlet prior)")
plt.suptitle("Keyness: words over-used in each target-language set (top 20 by z)", fontsize=11); plt.tight_layout(); plt.savefig(FIG / "fig9a_keyness_lang.png", dpi=160); plt.show()

## 3. 2026년 대 2023~2025년 — 세계 전체에서 새로 늘어난 말

- 출력: `analysis/output/tables/keyness_2026_vs_prior.csv`
- 읽는 법: 2026년 논문에서 상대적으로 많이 쓰인 말(상위)과 줄어든 말(하위). 07의 연어, 09의 버스트와 겹치는지 본다.


In [ ]:
ids_26 = allc[allc.year == 2026].id; ids_pr = allc[allc.year < 2026].id
c26, d26 = counts_of(ids_26); cpr, dpr = counts_of(ids_pr)
t26 = log_odds(c26, sum(c26.values()), cpr, sum(cpr.values())); t26["df_2026"] = [d26.get(w, 0) for w in t26.index]; t26["df_prior"] = [dpr.get(w, 0) for w in t26.index]
t26["pct_docs_2026"] = (100 * t26.df_2026 / len(ids_26)).round(2); t26["pct_docs_prior"] = (100 * t26.df_prior / len(ids_pr)).round(2)
t26 = t26.sort_values("z", ascending=False); t26.to_csv(TAB / "keyness_2026_vs_prior.csv")
print("2026년에 늘어난 말:", ", ".join(f"{w}(z={r.z:.0f}, {r.pct_docs_2026:.1f}%←{r.pct_docs_prior:.1f}%)" for w, r in t26.head(30).iterrows()))
print("\n2026년에 줄어든 말:", ", ".join(f"{w}(z={r.z:.0f}, {r.pct_docs_2026:.1f}%←{r.pct_docs_prior:.1f}%)" for w, r in t26.tail(20).iloc[::-1].iterrows()))

## 4. 가중 키워드 동시출현 연결망(co-word network) — 전체 기간과 2023~2025 대 2026

- 계산: 표제어(불용어·도메인 일반어·언어 이름 제외)와 인접 표제어 2-gram 가운데 문서 빈도 상위 60개를 노드로 둔다. 간선 가중치 = 두 키워드가 같은 논문에 함께 나타난 논문 수, 가중치 상위 250개 간선만 그린다. 노드 크기 ∝ 문서 빈도. 참고 논문(류지혜 외 2025)의 그림 2와 같은 구성이며, 옛 노트북 04의 정규식 토큰 판을 spaCy 표제어·2023-01~2026-09 LLM 에이전트 20,107편으로 바꾼 것이다.
- 배치는 spring layout(seed 42)이며 거리에 뜻을 두지 않는다. 전체 기간 한 장, 2023~2025와 2026(1~9월)을 나란히 한 장.
- 출력: `analysis/output/tables/coword_top60_{all,2023_2025,2026}.csv`, `analysis/output/figures/fig9e_coword_network_all.png`, `fig9f_coword_network_2023_2025_vs_2026.png`


In [ ]:
import networkx as nx
GENERIC = {"introduce", "present", "presents", "significant", "potential", "including", "achieves", "achieve", "extensive", "current", "provide",
           "provides", "evaluate", "effectiveness", "efficiency", "key", "critical", "challenge", "challenges", "challenging", "dynamic",
           "efficient", "compared", "effective", "time", "robust", "recent", "multiple", "diverse", "process", "problem", "problems", "research",
           "information", "complex", "state-of-the-art", "analysis", "applications", "application", "important", "enable", "enables", "leverage",
           "leverages", "improve", "improves", "improved", "outperforms", "existing", "propose", "proposes", "paper", "design", "designed",
           "develop", "developed", "ability", "capabilities", "capability", "scenarios", "real-world", "settings", "setting", "wide", "range",
           "used", "enabling", "future", "limited", "comprehensive", "effectively", "specifically", "remains", "need", "furthermore", "like",
           "traditional", "development", "limitations", "enhance", "generate", "learn", "space", "context", "features", "network", "networks",
           "algorithm", "algorithms", "neural", "deep", "strategies", "understanding", "interactions", "interaction", "actions", "behavior",
           "control", "optimization", "detection", "accuracy", "dataset", "datasets", "benchmark", "benchmarks", "evaluation", "generation"}
NET_STOP = STOP | LANG_WORDS | GENERIC | {"require", "address", "remain", "high", "new", "different", "various", "specific", "well", "work", "approach", "exist", "include", "produce", "reduce", "support", "increasingly", "strong", "single", "rate", "gap", "ground", "level", "real", "offer", "drive", "datum", "experiment", "result", "show", "find", "highlight", "demonstrate", "compare", "base", "ensure", "allow", "make", "large", "small", "low", "open", "long", "source", "quality", "accuracy", "generate", "generation", "build", "context", "domain", "world", "scale", "general", "novel", "overall", "finally", "additionally", "study"}
def doc_units(toks):
    u = set(w for w in toks if w not in NET_STOP)
    u |= set(f"{a}_{b}" for a, b in zip(toks, toks[1:]) if a not in NET_STOP and b not in NET_STOP)
    return u
def coword(ids, top=60, edges=250):
    units = {i: doc_units(lem[i]) for i in ids}
    dfc = Counter(); [dfc.update(u) for u in units.values()]
    # 2-gram이 상위에 들면 그 구성 단어 하나짜리는 뺀다(중복 노드 억제)
    keep = []
    for w, n in dfc.most_common(400):
        if "_" in w and all(p not in keep for p in w.split("_")) or "_" not in w and not any(w in k.split("_") for k in keep):
            keep.append(w)
        if len(keep) == top: break
    kset = set(keep); co = Counter()
    for u in units.values():
        ks = sorted(u & kset)
        for a in range(len(ks)):
            for b in range(a + 1, len(ks)): co[(ks[a], ks[b])] += 1
    G = nx.Graph(); [G.add_node(w, df=dfc[w]) for w in keep]
    for (a, b), n in co.most_common(edges): G.add_edge(a, b, weight=n)
    return G, dfc
def draw(G, ax, title):
    G = G.subgraph([n for n in G if G.degree(n) > 0]).copy()                      # 상위 250 간선에 들지 못한 고립 노드는 그리지 않는다
    wmax = max(d["weight"] for _, _, d in G.edges(data=True))
    for a, b, d in G.edges(data=True): d["w01"] = d["weight"] / wmax                 # 끌림 세기는 0~1로 정규화(원 가중치는 수천 편이라 배치가 뭉친다)
    pos = nx.spring_layout(G, seed=42, k=1.6 / np.sqrt(G.number_of_nodes()), weight="w01", iterations=200); sizes = [G.nodes[w]["df"] for w in G]; mx = max(sizes)
    nx.draw_networkx_edges(G, pos, ax=ax, width=[0.3 + 3 * G[a][b]["w01"] for a, b in G.edges()], edge_color="#999999", alpha=0.5)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[300 + 2500 * s / mx for s in sizes], node_color="#48C9B0", alpha=0.9)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_weight="bold"); ax.set_title(title, fontsize=10); ax.axis("off")
sets_net = {"all": allc.id, "2023_2025": allc[allc.year <= 2025].id, "2026": allc[allc.year == 2026].id}
Gs = {}
for name, ids in sets_net.items():
    G, dfc = coword(ids); Gs[name] = G
    pd.DataFrame({"keyword": list(G.nodes), "doc_freq": [G.nodes[w]["df"] for w in G.nodes], "doc_pct": [round(100 * G.nodes[w]["df"] / len(ids), 1) for w in G.nodes]}).sort_values("doc_freq", ascending=False).to_csv(TAB / f"coword_top60_{name}.csv", index=False)
    print(name, f"n={len(ids):,}", "| 상위 15:", ", ".join(f"{w}({G.nodes[w]['df']})" for w in sorted(G.nodes, key=lambda w: -G.nodes[w]["df"])[:15]))
fig, ax = plt.subplots(figsize=(13, 10)); draw(Gs["all"], ax, f"Weighted keyword co-occurrence network, LLM-agent papers 2023-01 to 2026-09 (n={len(allc):,}; top-60 keywords, top-250 edges)")
plt.tight_layout(); plt.savefig(FIG / "fig9e_coword_network_all.png", dpi=150, bbox_inches="tight"); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(22, 10))
draw(Gs["2023_2025"], axes[0], f"2023-2025 (n={len(sets_net['2023_2025']):,})"); draw(Gs["2026"], axes[1], f"2026 Jan-Sep (n={len(sets_net['2026']):,})")
plt.tight_layout(); plt.savefig(FIG / "fig9f_coword_network_2023_2025_vs_2026.png", dpi=150, bbox_inches="tight"); plt.show()
only26 = sorted(set(Gs["2026"].nodes) - set(Gs["2023_2025"].nodes)); only_pre = sorted(set(Gs["2023_2025"].nodes) - set(Gs["2026"].nodes))
print("2026 연결망에만 있는 노드:", only26); print("2023~2025 연결망에만 있는 노드:", only_pre)
coword_summary = {"top60": {k: [{"keyword": w, "df": int(G.nodes[w]["df"])} for w in sorted(G.nodes, key=lambda w: -G.nodes[w]["df"])] for k, G in Gs.items()}, "only_2026": only26, "only_2023_2025": only_pre, "edges": 250}

## 5. 정본 저장

In [ ]:
out = {"generated": TODAY, "notebook": "analysis/notebooks/08_keyness.ipynb", "coword_network": coword_summary, "method": "log-odds ratio with informative Dirichlet prior (Monroe et al. 2008), alpha0=1000, vocab min count 20, language names excluded",
       "n_vocab": len(vocab), "set_sizes": {l: int(allc[f"lang_{l}"].sum()) for l in LANGS}, "n_2026": int(len(ids_26)), "n_prior": int(len(ids_pr)),
       "lang_top": {l: KL[l].head(25).reset_index().to_dict(orient="records") for l in LANGS}, "lang_bottom": {l: KL[l].tail(10).iloc[::-1].reset_index().to_dict(orient="records") for l in LANGS},
       "y2026_top": t26.head(40).reset_index().to_dict(orient="records"), "y2026_bottom": t26.tail(25).iloc[::-1].reset_index().to_dict(orient="records")}
p = VER / f"meta_keyness_{TODAY}.json"; p.write_text(json.dumps(out, ensure_ascii=False, indent=1, default=float)); print("saved", p)